# ENGRAM v5: SEEN line (last) + meaning search + image search

What changed since the paper (prompt **lines-v5**):
1. The big model also writes a **SEEN:** line with up to 5 objects in view, as its LAST line (generation stops there),
   so *"where is my bicycle?"* can be found even when nothing was done with it.
   (v4 put SEEN first: the model got 60% slower and wrote half as many STATE lines, so v5 moves it to the end.)
2. **Meaning search:** diary lines and questions become vectors (small text model), so *"cooker"* finds *"stove"*.
3. **Image search:** a CLIP fingerprint of EVERY 4-s window, also the ones the trigger skipped.

Two tests, one after the other:
- **Part A, EPIC-KITCHENS (~1.5 h):** 20 test minutes. Then an automatic **go / no-go check**: the STATE lines must be back
  (≥ 1.8 per window; v3 had 2.26) and the big model must not be much slower (≤ 8 s per window; v3 took 6.6 s).
  If the check fails, Part B is skipped (saves ~6 GPU hours).
- **Part B, Ego4D (~6 h):** 20 **new** clips (the 63 clips used before are skipped). Every search part is scored separately.

**Settings (right panel):** Accelerator **GPU T4** · Internet **On** · Input: dataset **mnemofyp** (latest version)
· **Add-ons → Secrets**: tick `Access Key ID`, `Secret Access Key`, `Region`.

**How to run:** top right **Save Version** → **Save & Run All (Commit)** → Save. About 7.5 hours; close the browser.
Download `v5_results.zip` from the finished version's **Output** (Part A alone is also saved as `v5_epic_results.zip`).

In [ ]:
RUN_EPIC = True     # Part A: EPIC-KITCHENS (~1.5 h)
RUN_EGO4D = True    # Part B: Ego4D (~6 h). Short on GPU hours? Run one part per version.


## 1. Code + packages (~3 min)

In [ ]:
import os, shutil, glob, sys, subprocess
WORK = '/kaggle/working'
CODE = f'{WORK}/mnemo'
code_dir = code_zip = None
for root, dirs, files in os.walk('/kaggle/input', followlinks=True):
    if 'mnemo.py' in files and 'modules' in dirs:
        code_dir = root
    if 'mnemo_code.zip' in files:
        code_zip = os.path.join(root, 'mnemo_code.zip')
print('code folder:', code_dir, '| zip:', code_zip)
assert code_dir or code_zip, 'Code not found: add the mnemofyp dataset (Add Input) and wait until it is attached'
shutil.rmtree(CODE, ignore_errors=True)
if code_dir:
    shutil.copytree(code_dir, CODE)
else:
    os.makedirs(CODE)
    !cd "{CODE}" && unzip -q "{code_zip}"
%cd {CODE}
assert os.path.exists('modules/text_embed.py'), 'OLD code: upload the new mnemo_code.zip as a new version of mnemofyp'
!pip install -q -U "transformers>=4.57" accelerate mediapipe ultralytics ego4d "open_clip_torch==3.3.0" "sentence-transformers==5.1.1"
import torch, numpy, transformers
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE: switch Accelerator to GPU T4!')
print('transformers', transformers.__version__, '| numpy', numpy.__version__)

## 2. Ego4D access + questions (~1 min)
The keys come from Kaggle Secrets and are written to `~/.aws` (not to the output).

In [ ]:
if RUN_EGO4D:
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    aws = os.path.expanduser('~/.aws')
    os.makedirs(aws, exist_ok=True)
    with open(f'{aws}/credentials', 'w') as f:
        f.write('[default]\n'
                f'aws_access_key_id = {s.get_secret("Access Key ID")}\n'
                f'aws_secret_access_key = {s.get_secret("Secret Access Key")}\n')
    with open(f'{aws}/config', 'w') as f:
        f.write(f'[default]\nregion = {s.get_secret("Region")}\n')
    os.chmod(f'{aws}/credentials', 0o600)

    E4 = '/tmp/ego4d'                 # NOT in /kaggle/working: videos must not end up in the output
    !ego4d --output_directory {E4} --datasets annotations --benchmarks EM -y > /tmp/ego4d_annotations.log 2>&1
    NLQ_FILE = glob.glob(f'{E4}/**/nlq_val.json', recursive=True)[0]
    print('questions file:', NLQ_FILE)

## 3. Start downloading the EPIC test minutes (background)

In [ ]:
if RUN_EPIC:
    import urllib.request
    EK = f'{WORK}/epic'
    CLIPS_DIR = f'{WORK}/clips_v2'
    os.makedirs(EK, exist_ok=True); os.makedirs(CLIPS_DIR, exist_ok=True)
    A = 'https://raw.githubusercontent.com/epic-kitchens/epic-kitchens-100-annotations/master'
    D = 'https://raw.githubusercontent.com/epic-kitchens/epic-kitchens-download-scripts/master/data'
    for url in [f'{A}/EPIC_100_train.csv', f'{A}/EPIC_100_verb_classes.csv', f'{A}/EPIC_100_noun_classes.csv',
                f'{A}/EPIC_100_video_info.csv', f'{D}/epic_55_splits.csv']:
        dst = f'{EK}/{url.rsplit("/", 1)[1]}'
        if not os.path.exists(dst):
            urllib.request.urlretrieve(url, dst)
    ANN = f'{EK}/EPIC_100_train.csv'
    N_CLIPS = 20
    TEST_P = ['P02', 'P06', 'P08', 'P14', 'P26']
    SEED = 7            # test people P02 P06 P08 P14 P26 (same minutes as the v4 run, for a v4 → v5 comparison)
    FETCH_LOG = f'{WORK}/fetch.log'
    FETCH = subprocess.Popen([sys.executable, '-m', 'tools.fetch_epic_windows', '--data', EK, '--out', CLIPS_DIR,
                              '--select', 'random', '--windows', str(N_CLIPS), '--window-s', '60', '--workers', '8',
                              '--participants', *TEST_P, '--seed', str(SEED)],
                             cwd=CODE, stdout=open(FETCH_LOG, 'w'), stderr=subprocess.STDOUT)
    print('download started in the background → continue with step 3')

## 4. Load the models (~5 min)

In [ ]:
import time, json
import config as C
from modules.vlm_logger import VLMDescriber
from modules.hoi_gate import HOIGate
from modules.landmarks import ClipEmbedder
from modules.text_embed import TextEmbedder
t = time.time()
describer = VLMDescriber(C.VLM_MODEL)
gate = HOIGate('models/hoi_tcn.pt')
clip = ClipEmbedder()          # image fingerprints
text = TextEmbedder()          # meaning search
print(f'loaded in {time.time() - t:.0f}s | prompt {C.VLM_PROMPT_VERSION} | SEEN line {C.VLM_SEEN} | trigger threshold {C.GATE_THRESHOLD}')
assert C.VLM_PROMPT_VERSION == 'lines-v5'

## 5. Part A: EPIC-KITCHENS benchmark (~1h15) + whole-morning questions (~10 min)

In [ ]:
RESULTS = f'{WORK}/v5_results'
if RUN_EPIC:
    FETCH.wait()
    print(open(FETCH_LOG).read()[-600:])
    CLIPS = sorted(glob.glob(f'{CLIPS_DIR}/*_w*.mp4'))
    print(len(CLIPS), 'clips')
    from evaluation.benchmark import run_benchmark
    from evaluation.long_qa import run_long_qa
    from IPython.display import Markdown, display
    EPIC_RESULTS = f'{RESULTS}/epic'
    results, summary = run_benchmark(CLIPS, describer, gate, ANN, EK, EPIC_RESULTS)
    display(Markdown(summary))
    long_res, when, long_md = run_long_qa(CLIPS, describer, ANN, EK, EPIC_RESULTS, n_yes=48, n_no=24)
    display(Markdown(long_md))
    shutil.make_archive(f'{WORK}/v5_epic_results', 'zip', EPIC_RESULTS)     # saved now in case Part B stops
    shutil.rmtree(CLIPS_DIR, ignore_errors=True)

## Go / no-go check before Part B

In [ ]:
import re, numpy as np
if RUN_EPIC:
    ws = [w for f in glob.glob(f'{EPIC_RESULTS}/vlm_cache/*__{C.VLM_PROMPT_VERSION}.json')
          for w in json.load(open(f))['windows'] if w['text']]
    state_rate = np.mean([len(re.findall(r'^\s*STATE\s*:', w['text'], re.M | re.I)) for w in ws])
    sec = np.mean([w['dt'] for w in ws])
    did = [l for w in ws for l in w['text'].splitlines() if l.strip().upper().startswith('DID')]
    no_pipe = np.mean(['|' not in l for l in did]) if did else 0.0
    GO = state_rate >= 1.8 and sec <= 8.0
    print(f'STATE lines per window {state_rate:.2f} (v3 2.26, v4 0.96) | big model {sec:.1f} s per window (v3 6.6, v4 10.5)'
          f' | DID lines without "|" {no_pipe:.0%} (v3 9%, v4 58%)')
    print('GO: Part B runs' if GO else 'NO-GO: the SEEN line still hurts, Part B is skipped (send me the results)')
    RUN_EGO4D = RUN_EGO4D and GO

## 6. Part B: download 20 new Ego4D clips (~72 MB each)
The first 63 clips of this selection were used before (3 to fix the search, 20 for the first test, 20 for the trigger test, 20 for the v4 test), so they are skipped.

In [ ]:
if RUN_EGO4D:
    N_FULL, N_EXTRA, SKIP, SEED = 20, 0, 63, 0
    !python -m tools.fetch_ego4d --nlq "{NLQ_FILE}" --out {E4} --clips {N_FULL} --extra {N_EXTRA} --seed {SEED} --skip {SKIP}
    SEL = json.load(open(f'{E4}/selected.json'))
    FULL = [c for c in SEL if c['full']]
    EXTRA = [c for c in SEL if not c['full']]
    print(len(FULL), 'full clips,', len(EXTRA), 'trigger-only clips')

## 7. Part B: Ego4D questions with every search part (~6 h)
Finished clips are cached.

In [ ]:
if RUN_EGO4D:
    from evaluation.ego4d_nlq import load_nlq, run_nlq
    from IPython.display import Markdown, display
    EGO_RESULTS = f'{RESULTS}/ego4d'
    nlq = load_nlq(NLQ_FILE)
    ego_results, ego_md = run_nlq(FULL, nlq, describer, gate, EGO_RESULTS, embedder=clip, text=text)
    display(Markdown(ego_md))

## 8. Save
Download `v4_results.zip` (tables, every answer, diaries, cached model text and image fingerprints; no video).

In [ ]:
shutil.make_archive(f'{WORK}/v5_results', 'zip', RESULTS)
os.chdir(WORK)
from IPython.display import FileLink
display(FileLink('v5_results.zip'))